Setup, load environment and libs

In [2]:
from dotenv import load_dotenv
load_dotenv()  # reads .env file
import pandas as pd
import csv
import io
import os
import hashlib

from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_mistralai import ChatMistralAI

## Build a chain

In [3]:
template = """
Does the description of this job "{description}"
fit the candidate's profile "{profile}? 
For each of the following aspects: 
* technical skills set required, 
* work area of the organization, 
* role position within the organization, 
* geographical location, 
Give me a score like: 
1.Perfect fit, 2. Good Fit, 3. Probable Fit, 4. Poor Fit, 5. Not a Fit.
make the score an integer in the range 1 to 5.
"""
prompt_template = PromptTemplate.from_template(template)

Create the analysis chain

In [4]:
def format_prompt(inputs):
    return prompt_template.format(
        description=inputs["description"], 
        profile=inputs["profile"])

format_runnable = RunnableLambda(format_prompt)

llm = ChatMistralAI(model="mistral-small-latest")  # real model, not LLM()

analysis_chain = format_runnable | llm

Create the formating chain

In [5]:
# 1. Define the structure you want extracted from the analysis
class Row(BaseModel):
    aspect: str
    score: int = Field(ge=1, le=5)
    reasoning: str

class Table(BaseModel):
    rows: list[Row]

# 2. Second model call: constrained to this exact schema
structured_llm = llm.with_structured_output(Table)

## Load the crawled data

In [12]:
bht_df = pd.read_excel(os.path.join("../jobs", "jobs_BHT_2026-11-06.xlsx"))
tu_df = pd.read_excel(os.path.join("../jobs", "jobs_TU_2026-11-06.xlsx"))
#an_df = pd.read_excel(os.path.join("../jobs", "jobs_Arbeitnow_2026-11-06.xlsx"))
fu_df = pd.read_excel(os.path.join("../jobs", "jobs_FU_2026-11-07.xlsx"))
hu_df = pd.read_excel(os.path.join("../jobs", "jobs_HU_2026-11-07.xlsx"))

In [13]:
bht_df.head()

,id,posted,company,title,location,is_research,description,url
0,b122981f7bb8fa5d,2026-11-06 11:56,Institut für Fabrikanlagen und Logistik (IFA),Wissenschaftliche Mitarbeit (Promotionsstelle)...,NaN,True,"Wir bieten Ihnen die Möglichkeit, sich in ansp...",https://stellenticket.bht-berlin.de/en/job-off...
1,c8a2e98cc95c03b6,2026-11-06 11:40,Institut für Transport- und Automatisierungste...,Wissenschaftliche Mitarbeit im Bereich Entwick...,NaN,True,Sie verantworten ein anwendungsnahes Forschung...,https://stellenticket.bht-berlin.de/en/job-off...
2,a1f9c8b74406aab8,2026-11-06 09:38,"Fakultät Wirtschaftswissenschaften, Startup-S...",wiss. Mitarbeiterin bzw. Mitarbeiter (m/w/d)\n...,NaN,True,Initiierung und Coaching von Unternehmensgründ...,https://stellenticket.bht-berlin.de/en/job-off...
3,657dc7ba07b8244d,2026-11-05 13:44,"Faculty II - Mathematics and Natural Sciences,...",Research Assistant - 0.75 working-time,NaN,True,You will contribute to the development of phys...,https://stellenticket.bht-berlin.de/en/job-off...
4,d2804c3bbc7a78dd,2026-11-05 08:31,"Fakultät V - Verkehrs- und Maschinensysteme, I...",Wiss. Mitarbeiter*in (d/m/w) - 1. Qualifizieru...,NaN,True,Mitarbeit und Unterstützung in Forschung und L...,https://stellenticket.bht-berlin.de/en/job-off...


## Get an assessment for each job 

In [8]:
profile = """
I am looking for a mid to senior level Wissenschaftlicher Mitarbeiter 
(or alternatively: Informatiker, Data Scientist, Software Engineer) role in Education and Science 
(bioacoustics, ecology, climate, nature, monitoring, museum collection management, libraries), 
based in Berlin, Potsdam or Leipzig, with partly from home work arrangements. 
I bring data management (Postgres, SQL, ETL Pipelines using Airflow), Docker, research software engineering (Python, R) and machine learning (Keras, LangChain) skills, 
and I seek a position offering posibilities across research and development.
"""

In [9]:
def to_csv(source_name, job_id, table):
    path=os.path.join("../assessments", f"{job_id}_assessment.csv")
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["JobID", "Source",  "Aspect", "Score", "Reasoning"])
        for row in table.rows:
            writer.writerow([job_id, source_name, row.aspect, row.score, row.reasoning])
    return path

def assess(source_name, job_df):
    for i,row in job_df.iterrows():
        try:
            job_id = row.id
            description = row.description
            result = analysis_chain.invoke({"description": description, "profile": profile})
            table = structured_llm.invoke("Extract the assessment from this text, one row per aspect:\n\n" + result.content)
            to_csv(source_name, job_id, table)
        except(e):
            print(f"Skipping {job_id}")


In [9]:
assess("BHT", bht_df)

ValidationError: 1 validation error for Table
rows.3.score
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/int_type

In [10]:
assess("TUB", tu_df)

In [11]:
# assess("ArbeitNow", an_df)

In [11]:
assess("FU", fu_df)

In [14]:
assess("HU", hu_df)